# GeoPulse Toronto: Neighbourhood Analysis

**Decision Question:** Which Toronto neighbourhoods look underserved on transit access + housing pressure relative to 311 demand and reported crime — and where should a city analyst look first?

**Author:** Akash Gupta (York University CS)  
**Data Source:** City of Toronto Open Data Portal (CKAN API)

## Setup and Imports

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

print("✓ Imports loaded")

## 1. Load Processed Data

Load the processed neighbourhood analysis data created by `scripts/process_data.py`

In [ ]:
gdf = gpd.read_file('data/processed/neighbourhoods_analysis.geojson')

print(f"Loaded {len(gdf)} neighbourhoods")
print(f"Columns: {', '.join(gdf.columns)}")
gdf.head()

## 2. Data Quality Assessment

In [ ]:
# Check coverage
coverage = {
    'Crime Rate': gdf['crime_rate'].notna().sum() / len(gdf) * 100,
    'Transit Access': gdf['stop_density'].notna().sum() / len(gdf) * 100,
    'Housing Pressure': gdf['housing_pressure'].notna().sum() / len(gdf) * 100,
    '311 Requests': gdf['request_count'].notna().sum() / len(gdf) * 100,
}

print("Data Coverage by Layer:")
for layer, pct in coverage.items():
    print(f"  {layer}: {pct:.1f}%")

# Summary statistics
print("\nSummary Statistics:")
gdf[['crime_rate', 'stop_density', 'median_stop_dist', 'housing_pressure', 
     'request_rate', 'attention_score_normalized']].describe()

## 3. Exploratory Visualizations

### 3.1 Attention Score Distribution

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Attention score distribution
axes[0, 0].hist(gdf['attention_score_normalized'], bins=30, color='steelblue', edgecolor='black')
axes[0, 0].set_title('Attention Score Distribution', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Attention Score (0-100)')
axes[0, 0].set_ylabel('Count')

# Crime rate
axes[0, 1].hist(gdf['crime_rate'].dropna(), bins=30, color='coral', edgecolor='black')
axes[0, 1].set_title('Crime Rate Distribution', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Crime Rate (per 1k residents)')
axes[0, 1].set_ylabel('Count')

# Transit access
axes[1, 0].hist(gdf['stop_density'].dropna(), bins=30, color='mediumseagreen', edgecolor='black')
axes[1, 0].set_title('Transit Stop Density', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Stops per km²')
axes[1, 0].set_ylabel('Count')

# Housing pressure
axes[1, 1].hist(gdf['housing_pressure'].dropna() * 100, bins=30, color='goldenrod', edgecolor='black')
axes[1, 1].set_title('Housing Pressure Distribution', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Housing Pressure (%)')
axes[1, 1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('notebooks/distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Saved: notebooks/distributions.png")

### 3.2 Correlation Analysis

In [ ]:
# Correlation matrix
corr_cols = ['crime_rate', 'stop_density', 'median_stop_dist', 
             'housing_pressure', 'request_rate', 'attention_score_normalized']
corr_matrix = gdf[corr_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlations', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('notebooks/correlations.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Saved: notebooks/correlations.png")

### 3.3 Spatial Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# Attention score map
gdf.plot(column='attention_score_normalized', cmap='RdYlGn_r', 
         legend=True, ax=axes[0, 0], edgecolor='black', linewidth=0.3)
axes[0, 0].set_title('Attention Score (Composite)', fontsize=12, fontweight='bold')
axes[0, 0].axis('off')

# Crime rate
gdf.plot(column='crime_rate', cmap='Reds', 
         legend=True, ax=axes[0, 1], edgecolor='black', linewidth=0.3)
axes[0, 1].set_title('Crime Rate (per 1k)', fontsize=12, fontweight='bold')
axes[0, 1].axis('off')

# Transit access
gdf.plot(column='stop_density', cmap='Greens', 
         legend=True, ax=axes[1, 0], edgecolor='black', linewidth=0.3)
axes[1, 0].set_title('Transit Stop Density (per km²)', fontsize=12, fontweight='bold')
axes[1, 0].axis('off')

# Housing pressure
gdf.plot(column='housing_pressure', cmap='YlOrRd', 
         legend=True, ax=axes[1, 1], edgecolor='black', linewidth=0.3)
axes[1, 1].set_title('Housing Pressure', fontsize=12, fontweight='bold')
axes[1, 1].axis('off')

plt.tight_layout()
plt.savefig('notebooks/maps.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Saved: notebooks/maps.png")

## 4. Top Priority Neighbourhoods

Identify the top 10 neighbourhoods that should receive attention based on the composite score.

In [ ]:
top_10 = gdf.nlargest(10, 'attention_score_normalized')[[
    'neighbourhood', 'attention_score_normalized', 'crime_rate', 
    'stop_density', 'median_stop_dist', 'housing_pressure', 'request_rate'
]].copy()

top_10.columns = ['Neighbourhood', 'Attention Score', 'Crime Rate', 
                  'Stop Density', 'Median Stop Dist (m)', 'Housing Pressure', '311 Rate']

print("Top 10 Priority Neighbourhoods:")
print("="*80)
for idx, row in top_10.iterrows():
    print(f"\n{row['Neighbourhood']}")
    print(f"  Attention Score: {row['Attention Score']:.1f}")
    print(f"  Crime Rate: {row['Crime Rate']:.1f} per 1k")
    print(f"  Stop Density: {row['Stop Density']:.1f} per km²")
    print(f"  Housing Pressure: {row['Housing Pressure']*100:.0f}%")
    print(f"  311 Rate: {row['311 Rate']:.1f} per 1k")

top_10

## 5. Methodological Notes

### Spatial Join Coverage
Report on the quality and coverage of spatial joins performed during data processing.

In [ ]:
with open('data/processed/summary_stats.json', 'r') as f:
    summary = json.load(f)

print("Data Quality Report:")
print("="*60)
print(f"Total Neighbourhoods: {summary['total_neighbourhoods']}")
print("\nJoin Coverage:")
for layer, coverage in summary['data_quality'].items():
    print(f"  {layer}: {coverage:.1f}%")

print("\n⚠️  Caveats:")
print("  - MAUP: Results sensitive to neighbourhood boundary definitions")
print("  - Ecological fallacy: Neighbourhood-level patterns ≠ individual outcomes")
print("  - Confounders: Socioeconomic status, development history not controlled")
print("  - No causality: This is descriptive analysis only")
print("  - Temporal: Data sources may reflect different time periods")

## 6. Export for Decision Memo

Export top findings for the decision memo.

In [ ]:
# Export top 10 for memo
top_10_dict = gdf.nlargest(10, 'attention_score_normalized')[[
    'neighbourhood', 'attention_score_normalized', 'crime_rate', 
    'stop_density', 'housing_pressure', 'request_rate'
]].to_dict('records')

with open('docs/top_neighbourhoods.json', 'w') as f:
    json.dump(top_10_dict, f, indent=2)

print("✓ Exported: docs/top_neighbourhoods.json")
print("\n📊 Analysis complete! Next step: Review decision memo in docs/decision_memo.md")